# 01 — Customer churn exploration

Quick DS-style notebook against the synthetic Telco-churn CSV. Trains a logistic
regression and a random forest, logs both runs to MLflow under the
`notebook-exploration` experiment, and prints a clickable URL to the better run.

Prereqs: the stack is up (`make up`) so MLflow is reachable at `http://mlflow:5000`,
and `data/raw/customer_churn.csv` exists (run `make generate-data` if not).

In [ ]:
# The pinned scipy-notebook image ships scikit-learn + pandas + numpy but
# NOT mlflow. Install it on first run; subsequent runs are a no-op.
import importlib.util, subprocess, sys
if importlib.util.find_spec('mlflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'mlflow==3.12.0', 'boto3==1.43.6'])

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://mlflow:5000')
EXPERIMENT_NAME = 'notebook-exploration'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f'MLflow tracking URI: {MLFLOW_TRACKING_URI}')
print(f'Experiment:          {EXPERIMENT_NAME}')

## Load the raw CSV

The CSV ships with whitespace `total_charges` for brand-new customers (tenure=0).
We treat blank-or-whitespace as NaN so we can impute below.

In [ ]:
CSV_PATH = Path('../data/raw/customer_churn.csv')
df = pd.read_csv(CSV_PATH, na_values=[' ', ''], skipinitialspace=True)
print(f'rows: {len(df):,}    cols: {len(df.columns)}')
df.head()

In [ ]:
# Light EDA — shape of the numeric columns + churn rate by contract.
df[['tenure', 'monthly_charges', 'total_charges']].describe()

In [ ]:
churn_by_contract = (
    df.assign(churned=(df['churn'] == 'yes').astype(int))
    .groupby('contract_type')['churned']
    .agg(['count', 'mean'])
    .rename(columns={'mean': 'churn_rate'})
    .sort_values('churn_rate', ascending=False)
)
churn_by_contract

## Preprocessing

* Impute blank `total_charges` with the median.
* One-hot encode the categoricals.
* Standardise the numerics (helps logistic regression).
* Hold out 20 % for the test set.

In [ ]:
TARGET = 'churn'
y = (df[TARGET] == 'yes').astype(int)
X = df.drop(columns=[TARGET, 'customer_id'])

numeric_cols = ['tenure', 'monthly_charges', 'total_charges']
categorical_cols = [c for c in X.columns if c not in numeric_cols]

preprocess = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline(steps=[
                ('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            'cat',
            Pipeline(steps=[
                ('impute', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_cols,
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'train: {X_train.shape}    test: {X_test.shape}')

## Train two models, log everything via autolog

In [ ]:
mlflow.sklearn.autolog(log_models=True, log_input_examples=False, silent=True)

def fit_and_score(name: str, estimator) -> dict:
    pipe = Pipeline(steps=[('prep', preprocess), ('clf', estimator)])
    with mlflow.start_run(run_name=name) as run:
        pipe.fit(X_train, y_train)
        proba = pipe.predict_proba(X_test)[:, 1]
        pred = (proba >= 0.5).astype(int)
        metrics = {
            'test_accuracy': accuracy_score(y_test, pred),
            'test_f1':       f1_score(y_test, pred),
            'test_roc_auc':  roc_auc_score(y_test, proba),
        }
        mlflow.log_metrics(metrics)
        mlflow.set_tag('model_family', name)
        return {'name': name, 'run_id': run.info.run_id, **metrics}

results = [
    fit_and_score('logistic_regression', LogisticRegression(max_iter=400, n_jobs=None)),
    fit_and_score('random_forest',       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
]
pd.DataFrame(results).set_index('name')

In [ ]:
# Pick the better model by ROC-AUC and print a clickable URL.
best = max(results, key=lambda r: r['test_roc_auc'])
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
ui_base = MLFLOW_TRACKING_URI.replace('mlflow:5000', 'localhost:5000')
run_url = f"{ui_base}/#/experiments/{experiment.experiment_id}/runs/{best['run_id']}"
print(f"best model: {best['name']}  (roc_auc={best['test_roc_auc']:.4f})")
print(f'run URL:    {run_url}')